# Module 1 — Papers (OpenAlex)

Builds the empirical paper series from the raw OpenAlex data. For every paper,
the novelty is the combination of its keywords and the explorer is its first
author; the notebook accumulates both over the event sequence and writes the
reduced series the rest of the pipeline reads.

**Input** — one parquet per subfield, `merged_df_subfield_{ID}.parquet`, in
`data/papers_openalex`, with columns `date`, `topics`, `keywords`, `authors`.

**Output** — into `outputs/`: `delta_rows_papers.csv`,
`openalex_combo_freq.csv`, `openalex_author_freq.csv`,
`reduced_openalex_{log,lin}.csv`, `openalex_author_dates_1980_2020.pkl` and
`openalex_intervals.pkl`.

The heavy work is done by `lib/module1_optimized.py`. Expect a long run.


In [1]:
# Repository bootstrap.
# The notebooks are meant to be run from the repository root; the walk keeps
# them working when a front-end starts the kernel in a subdirectory.
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "lib").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "lib") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "lib"))

DATA_DIR = REPO_ROOT / "data"          # raw inputs, you provide these
OUTPUT_DIR = REPO_ROOT / "outputs"     # derived data, auto-generated
FIGURE_DIR = REPO_ROOT / "figures"     # manuscript figures, versioned

import pandas as pd


from module1_optimized import build_papers_module1_outputs


## Parameters

`filter_empty_keywords = True` is the convention of the revised manuscript: a
paper with no keyword cannot produce a combination novelty, so counting it
would advance the event clock `t` without contributing to `D`. First authors
with more than `max_papers_per_year` papers in a single year are dropped.


In [2]:
# Read order of the published run. The subfield frames are concatenated in
# this order and then sorted by date with a stable sort, so the order decides
# how papers sharing a date are sequenced: keep it as is.
subfields = [14, 25, 37, 39, 54, 114, 131, 246, 110, 142, 164, 172, 191, 193, 206, 210, 216, 248]
start_year = 1980
end_year = 2020
filter_empty_keywords = True
max_papers_per_year = 100

papers_dir = DATA_DIR / 'papers_openalex'
output_dir = OUTPUT_DIR


## Build


In [3]:
meta = build_papers_module1_outputs(
    papers_dir=papers_dir,
    output_dir=output_dir,
    subfields=subfields,
    start_year=start_year,
    end_year=end_year,
    filter_empty_keywords=filter_empty_keywords,
    max_papers_per_year=max_papers_per_year,
)
pd.Series(meta)


[module1/papers] building outputs
[module1/papers] loading globally ordered papers frame
[papers/global] [1/18] reading merged_df_subfield_14.parquet


[papers/global] [2/18] reading merged_df_subfield_25.parquet


[papers/global] [3/18] reading merged_df_subfield_37.parquet


[papers/global] [4/18] reading merged_df_subfield_39.parquet


[papers/global] [5/18] reading merged_df_subfield_54.parquet


...

[module1/papers] final pass 4,500,000/5,171,735 | seen combos=1,669,165 | seen authors=1,173,521


[module1/papers] final pass 5,000,000/5,171,735 | seen combos=1,832,364 | seen authors=1,291,320


[module1/papers] writing csv/pkl outputs


[module1/papers] done


strategy                       global_frame_single_pass
staged_rows                                        None
total_rows_before_exclusion                     5193236
total_rows_after_exclusion                      5171735
excluded_authors                                     23
unique_keywords                                   35020
unique_keyword_combinations                     1889046
unique_first_authors                            1332990
filter_empty_keywords                              True
max_papers_per_year                                 100
dtype: object

## Check the outputs


In [4]:
display(pd.read_csv(output_dir / 'delta_rows_papers.csv').head())
display(pd.read_csv(output_dir / 'reduced_openalex_lin.csv').head())
display(pd.read_csv(output_dir / 'openalex_combo_freq.csv').head())


,delta_rows,num_first_authors
0,40286,30414
1,42976,53494
2,42815,72196
3,46259,90100
4,48190,107001


,years,num_unique_combinations,num_unique_authors,row_indices
0,1980,1,1,1
1,1980,402,467,518
2,1980,795,920,1035
3,1980,1130,1297,1552
4,1980,1472,1719,2069


,rank,combo,frequency
0,1,['Lattice (music)'],20415
1,2,['Dynamics'],17112
2,3,['Operator (biology)'],15198
3,4,['Line (geometry)'],14624
4,5,['Star (game theory)'],13601
